# A/B Test Analysis: Does a New Landing Page Increase Conversions?

**Question:** Does the new landing page (B) give a higher conversion rate than the old page (A)?

**Data:** [A/B testing dataset on Kaggle](https://www.kaggle.com/datasets/zhangluyuan/ab-testing), included in this repository as `ab_data.csv`.

**Plan:**
1. Load and clean the data
2. Baseline: compare the raw conversion rates
3. Two-proportion z-test and 95% confidence interval
4. Sample-size calculation
5. Checks on the method: A/A test, peeking and stability
6. Sanity checks
7. Conclusion

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# Settings
DATA_PATH = "ab_data.csv"
ALPHA = 0.05      # significance level: we accept a 5% chance of a false positive
POWER = 0.80      # 80% chance of detecting a real effect of MIN_LIFT
MIN_LIFT = 0.005  # smallest useful lift: 0.5 percentage points (e.g. 12.0% -> 12.5%)
SEED = 42         # fixed random seed so every run gives the same numbers

## 1. Load and clean the data

Two kinds of rows would make the comparison unfair:
- **Mismatched rows:** a `control` user who saw the new page, or a `treatment` user who saw the old page. We can't tell which version they really had.
- **Duplicate users:** each user must be counted once, otherwise the observations are not independent. We keep the first visit.

In [ ]:
def clean_data(df):
    """Remove mismatched rows and duplicate users. Returns (clean data, report)."""
    report = {"rows_raw": len(df)}

    matched = (
        ((df["group"] == "control") & (df["landing_page"] == "old_page"))
        | ((df["group"] == "treatment") & (df["landing_page"] == "new_page"))
    )
    report["rows_mismatched"] = int((~matched).sum())
    df = df[matched]

    duplicated = df["user_id"].duplicated(keep="first")
    report["duplicate_users_removed"] = int(duplicated.sum())
    df = df[~duplicated].reset_index(drop=True)

    report["rows_clean"] = len(df)
    return df, report


raw = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df, report = clean_data(raw)
report

## 2. Baseline: compare the raw conversion rates

The simplest possible answer: which group converted more often? The error bars show the 95% range for each rate.

In [ ]:
summary = df.groupby("group")["converted"].agg(users="count", conversions="sum", rate="mean")
summary

In [ ]:
rates = summary["rate"]
errors = 1.96 * np.sqrt(rates * (1 - rates) / summary["users"])  # 95% CI half-width

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Control (old page)", "Treatment (new page)"], rates,
       yerr=errors, capsize=8, color=["#7f8c8d", "#2e86c1"])
ax.set_ylabel("Conversion rate")
ax.set_ylim(0.10, 0.13)  # zoomed in so the small difference is visible
ax.set_title("Conversion rate by group (95% CI)")
plt.show()

## 3. Two-proportion z-test and confidence interval

- **H₀:** rate_B = rate_A
- **H₁:** rate_B ≠ rate_A (two-sided)

The **z-test** uses the *pooled* rate for the standard error, because under H₀ both groups share one true rate.
The **confidence interval** uses the *unpooled* standard error, because there we do not assume the rates are equal.

In [ ]:
def two_proportion_ztest(conv_a, n_a, conv_b, n_b):
    """Two-sided z-test for the difference between two proportions. Returns (z, p_value)."""
    p_a, p_b = conv_a / n_a, conv_b / n_b
    p_pool = (conv_a + conv_b) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    z = (p_b - p_a) / se
    p_value = 2 * norm.sf(abs(z))  # sf = 1 - cdf, the area in one tail
    return z, p_value


def diff_confidence_interval(conv_a, n_a, conv_b, n_b, alpha=0.05):
    """Confidence interval for rate_B - rate_A. Returns (diff, low, high)."""
    p_a, p_b = conv_a / n_a, conv_b / n_b
    diff = p_b - p_a
    se = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    z_crit = norm.ppf(1 - alpha / 2)  # 1.96 for a 95% interval
    return diff, diff - z_crit * se, diff + z_crit * se


conv_a, n_a = summary.loc["control", "conversions"], summary.loc["control", "users"]
conv_b, n_b = summary.loc["treatment", "conversions"], summary.loc["treatment", "users"]

z, p_value = two_proportion_ztest(conv_a, n_a, conv_b, n_b)
diff, ci_low, ci_high = diff_confidence_interval(conv_a, n_a, conv_b, n_b, ALPHA)

print(f"Difference (B - A): {diff * 100:+.3f} percentage points")
print(f"95% CI:             [{ci_low * 100:+.3f}, {ci_high * 100:+.3f}] percentage points")
print(f"z = {z:.3f}, p = {p_value:.4f}, significant at {ALPHA}: {p_value < ALPHA}")
print(f"Is a {MIN_LIFT * 100:.1f} pp lift still inside the CI? {ci_high >= MIN_LIFT}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.errorbar(diff, 0, xerr=[[diff - ci_low], [ci_high - diff]],
            fmt="o", capsize=8, color="#2e86c1")
ax.axvline(0, color="black", linestyle="--", label="No difference")
ax.axvline(MIN_LIFT, color="#c0392b", linestyle=":", label="Minimum useful lift")
ax.set_yticks([])
ax.set_xlabel("Treatment rate minus control rate")
ax.set_title("95% confidence interval for the difference")
ax.legend(loc="upper left", fontsize=8)
plt.show()

## 4. Sample size: was the test big enough?

Users needed **per group** to detect a lift of `MIN_LIFT` with a two-sided test:

$$n = \frac{(z_{\alpha/2} + z_{\beta})^2 \,\big(p_1(1-p_1) + p_2(1-p_2)\big)}{(p_2 - p_1)^2}$$

In [ ]:
def required_sample_size(baseline_rate, min_lift, alpha=0.05, power=0.80):
    """Users needed per group to detect a lift of min_lift."""
    p1, p2 = baseline_rate, baseline_rate + min_lift
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    n = (z_alpha + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2
    return int(np.ceil(n))


baseline_rate = conv_a / n_a
n_needed = required_sample_size(baseline_rate, MIN_LIFT, ALPHA, POWER)
n_have = int(summary["users"].min())
print(f"Users needed per group for a {MIN_LIFT * 100:.1f} pp lift: {n_needed:,}")
print(f"Users we have in the smaller group:        {n_have:,}")

In [ ]:
lifts = np.linspace(0.002, 0.02, 50)
needed = [required_sample_size(baseline_rate, l, ALPHA, POWER) for l in lifts]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(lifts * 100, needed, color="#2e86c1")
ax.axhline(n_have, color="#27ae60", linestyle="--", label="Users we have per group")
ax.set_yscale("log")
ax.set_xlabel("Lift to detect (percentage points)")
ax.set_ylabel("Users needed per group")
ax.set_title("Smaller lifts need far more users")
ax.legend()
plt.show()